# Notebook 3 - Interactive Product Price Dashboard (Plotly + ipywidgets)

**Project:** Web Scraping and Product Price Analysis System

**Run order:** `scraping/web_scraping.ipynb` (1) -> `analysis/product_price_analysis.ipynb` (2) -> **this notebook (3)**

**Input:** `data/scraped_product_data.csv` - the same file used by Notebook 2.

**Features:** KPI cards (count, average, minimum, maximum, median) - price distribution - average price by category -
products by price range - price vs rating - average price by vendor - top 10 expensive / cheapest products -
full dataset table - category / vendor / price filters - download of the filtered data and summary.

> **Status note:** this dashboard has **not been executed or tested yet**. Run every cell top to bottom in Google Colab
> and report any errors before you describe it as working in your report or README.

## 1. Install / import libraries

Plotly and ipywidgets are pre-installed in Colab. The command below only matters on a fresh local environment.

In [ ]:
# Uncomment if a library is missing (not normally needed in Colab):
# !pip install -q pandas plotly ipywidgets

In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from pathlib import Path

def find_project_root():
    """Return the repository root whether the notebook runs from a sub-folder
    (local / cloned repo) or from /content (a fresh Google Colab session)."""
    cwd = Path.cwd()
    return cwd.parent if cwd.name in {"scraping", "analysis", "dashboard"} else cwd

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "scraped_product_data.csv"
XLSX_PATH = DATA_DIR / "scraped_product_data.xlsx"
print("Project root :", PROJECT_ROOT)
print("Data folder  :", DATA_DIR)

EXPORT_DIR = PROJECT_ROOT / "dashboard_exports"   # ignored by Git
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Colab needs this line for ipywidgets to display correctly (skipped elsewhere).
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

## 2. Load and clean the data

In [ ]:
if not CSV_PATH.exists() and IN_COLAB:
    from google.colab import files
    print("scraped_product_data.csv not found - please upload it now.")
    for name, content in files.upload().items():
        if name.endswith(".csv"):
            CSV_PATH.write_bytes(content)

if not CSV_PATH.exists():
    raise FileNotFoundError(f"{CSV_PATH} not found. Run scraping/web_scraping.ipynb first.")


def to_number(series):
    cleaned = series.astype(str).str.replace(r"[^0-9.\-]", "", regex=True)
    return pd.to_numeric(cleaned, errors="coerce")


raw = pd.read_csv(CSV_PATH)
needed = ["Product_Name", "Vendor", "Category", "Price", "Rating"]
missing = [c for c in needed if c not in raw.columns]
assert not missing, f"Missing columns in CSV: {missing}"

data = raw.copy()
data["Price"] = to_number(data["Price"])
data["Rating"] = to_number(data["Rating"])
data = data.dropna(subset=["Price"]).reset_index(drop=True)

# Same equal-width Low/Medium/High split as Notebook 2.
RANGE_LABELS = ["Low", "Medium", "High"]
data["Price_Range"] = pd.cut(data["Price"], bins=3, labels=RANGE_LABELS)

print(f"Loaded {len(raw)} rows, {len(data)} rows have a valid price.")

## 3. Helper functions (filtering and summary)

In [ ]:
TABLE_COLUMNS = [c for c in ["ID", "Product_Name", "Vendor", "Category", "Price",
                             "Rating", "Stock", "SKU", "Variants", "Price_Range"]
                 if c in data.columns]


def filter_data(df, categories, vendors, price_min, price_max):
    """Apply the three dashboard filters. Empty selections mean 'no filter'."""
    out = df[(df["Price"] >= price_min) & (df["Price"] <= price_max)]
    if categories:
        out = out[out["Category"].isin(categories)]
    if vendors:
        out = out[out["Vendor"].isin(vendors)]
    return out


def build_summary(df):
    """Summary table for the (filtered) data."""
    if df.empty:
        return pd.DataFrame({"Metric": ["Total Products"], "Value": [0]})
    rating_df = df.dropna(subset=["Rating"])
    corr = rating_df["Price"].corr(rating_df["Rating"]) if len(rating_df) > 1 else float("nan")
    return pd.DataFrame({
        "Metric": ["Total Products", "Average Price", "Minimum Price", "Maximum Price",
                   "Median Price", "Price-Rating Correlation"],
        # dtype=object keeps the count as a whole number (67, not 67.0).
        "Value": pd.Series([len(df), round(df["Price"].mean(), 2), round(df["Price"].min(), 2),
                            round(df["Price"].max(), 2), round(df["Price"].median(), 2),
                            round(corr, 4) if pd.notna(corr) else "n/a"], dtype=object),
    })


def kpi_card(title, value):
    return (f"<div style='flex:1;min-width:120px;padding:12px;margin:4px;border-radius:8px;"
            f"background:#f1f5f9;text-align:center;font-family:sans-serif'>"
            f"<div style='font-size:12px;color:#475569'>{title}</div>"
            f"<div style='font-size:22px;font-weight:bold;color:#0f172a'>{value}</div></div>")

## 4. Build the filter widgets

In [ ]:
all_categories = sorted(data["Category"].dropna().unique())
all_vendors = sorted(data["Vendor"].dropna().unique())
price_low, price_high = float(data["Price"].min()), float(data["Price"].max())

category_select = widgets.SelectMultiple(options=all_categories, value=tuple(all_categories),
                                         description="Category", rows=6,
                                         layout=widgets.Layout(width="260px"))
vendor_select = widgets.SelectMultiple(options=all_vendors, value=tuple(all_vendors),
                                       description="Vendor", rows=6,
                                       layout=widgets.Layout(width="260px"))
price_slider = widgets.FloatRangeSlider(value=[price_low, price_high], min=price_low, max=price_high,
                                        step=max((price_high - price_low) / 200, 0.01),
                                        description="Price", readout_format=".2f",
                                        layout=widgets.Layout(width="420px"))
apply_button = widgets.Button(description="Apply filters", button_style="primary")
reset_button = widgets.Button(description="Reset filters")
download_data_button = widgets.Button(description="Download filtered data (CSV)")
download_summary_button = widgets.Button(description="Download summary (CSV)")

kpi_out = widgets.Output()
overview_out = widgets.Output()
charts_out = widgets.Output()
top_out = widgets.Output()
table_out = widgets.Output()
message_out = widgets.Output()

## 5. Render the dashboard

In [ ]:
current = {"df": data}   # the most recently filtered data (used by the download buttons)


def render(_=None):
    df = filter_data(data, category_select.value, vendor_select.value,
                     price_slider.value[0], price_slider.value[1])
    current["df"] = df
    for out in (kpi_out, overview_out, charts_out, top_out, table_out):
        out.clear_output(wait=True)

    if df.empty:
        with kpi_out:
            display(HTML("<b>No products match the selected filters.</b>"))
        return

    summary = build_summary(df)
    with kpi_out:
        cards = "".join(kpi_card(m, v) for m, v in zip(summary["Metric"][:5], summary["Value"][:5]))
        display(HTML(f"<div style='display:flex;flex-wrap:wrap'>{cards}</div>"))

    with overview_out:
        display(summary)
        fig = px.histogram(df, x="Price", nbins=20, title="Product Price Distribution")
        fig.update_layout(yaxis_title="Number of Products")
        fig.show()

    with charts_out:
        by_cat = df.groupby("Category", as_index=False)["Price"].mean().sort_values("Price", ascending=False)
        px.bar(by_cat, x="Category", y="Price", title="Average Price by Category",
               labels={"Price": "Average Price"}).show()

        by_range = (df["Price_Range"].value_counts().reindex(RANGE_LABELS).fillna(0)
                    .rename_axis("Price_Range").reset_index(name="Products"))
        px.bar(by_range, x="Price_Range", y="Products", title="Products by Price Range").show()

        px.scatter(df.dropna(subset=["Rating"]), x="Price", y="Rating",
                   hover_data=["Product_Name", "Vendor", "Category"],
                   title="Price vs Rating").show()

        by_vendor = df.groupby("Vendor", as_index=False)["Price"].mean().sort_values("Price", ascending=False)
        px.bar(by_vendor, x="Vendor", y="Price", title="Average Price by Vendor",
               labels={"Price": "Average Price"}).show()

    show = [c for c in TABLE_COLUMNS if c != "Price_Range"]
    with top_out:
        print("Top 10 most expensive products")
        display(df.nlargest(10, "Price")[show].reset_index(drop=True))
        print("Top 10 cheapest products")
        display(df.nsmallest(10, "Price")[show].reset_index(drop=True))

    with table_out:
        print(f"{len(df)} products (filtered)")
        with pd.option_context("display.max_rows", 600, "display.max_columns", None):
            display(df[TABLE_COLUMNS].reset_index(drop=True))


def reset(_=None):
    category_select.value = tuple(all_categories)
    vendor_select.value = tuple(all_vendors)
    price_slider.value = [price_low, price_high]
    render()


def deliver(df, filename):
    """Save to dashboard_exports/ and, in Colab, trigger a browser download."""
    path = EXPORT_DIR / filename
    df.to_csv(path, index=False)
    with message_out:
        message_out.clear_output(wait=True)
        print("Saved:", path)
    if IN_COLAB:
        from google.colab import files
        files.download(str(path))


def download_data(_=None):
    deliver(current["df"][TABLE_COLUMNS], "filtered_products.csv")


def download_summary(_=None):
    deliver(build_summary(current["df"]), "filtered_summary.csv")


apply_button.on_click(render)
reset_button.on_click(reset)
download_data_button.on_click(download_data)
download_summary_button.on_click(download_summary)

## 6. Show the dashboard

Change the filters and press **Apply filters**.

In [ ]:
filters = widgets.VBox([
    widgets.HBox([category_select, vendor_select]),
    price_slider,
    widgets.HBox([apply_button, reset_button, download_data_button, download_summary_button]),
    message_out,
])
tabs = widgets.Tab(children=[overview_out, charts_out, top_out, table_out])
for i, title in enumerate(["Overview", "Charts", "Top 10", "Dataset"]):
    tabs.set_title(i, title)

display(HTML("<h2 style='font-family:sans-serif'>Product Price Analysis Dashboard</h2>"))
display(filters, kpi_out, tabs)
render()

## 7. Notes

* Downloaded files are saved to `dashboard_exports/` (not tracked by Git) and, in Colab, sent to your browser.
* If the widgets do not appear in Colab, re-run the setup cell and make sure the custom widget manager line ran.
* Plotly charts opened inside tabs are drawn when you press **Apply filters** (they render on first load too).